# Green Roof Data Cleaning and Preparation
Load, filter, and prepare all buildings with green roofs, removing problematic parking structures that introduce bias.

## Preprocessing Workflow Overview
This notebook:
1. Loads the city-wide buildings dataset (buildings_berlin.shp).
2. Removes buildings where 'nutz' or 'geb_nutz' equals 'Tiefgarage' (underground parking).
3. Extracts green roof indicators.
4. Exports cleaned dataset as CSV and GPKG/SHP for further analysis.

### Step 1: Import libraries and set up paths
Load required packages and initialize project paths.

In [1]:
# Import required libraries
from pathlib import Path
import pandas as pd
import geopandas as gpd
import numpy as np

In [2]:
# Detect project root (notebook may run from /notebooks)
project_root = Path.cwd().resolve()
if not (project_root / "data").exists() and (project_root.parent / "data").exists():
    project_root = project_root.parent

# Define data input path
buildings_path = project_root / "data" / "green_roofs_berlin" / "buildings_berlin.shp"

# Validate required input file
if not buildings_path.exists():
    raise FileNotFoundError(f"Missing file: {buildings_path}")

### Step 2: Define geometry cleaning function
Helper function to repair invalid geometries and remove unrecoverable ones.

In [3]:
def clean_geometries(gdf: gpd.GeoDataFrame, name: str) -> gpd.GeoDataFrame:
    """Repair invalid geometries when possible and drop unrecoverable ones."""
    start = len(gdf)
    
    # Remove empty or missing geometries first
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    
    # Fix invalid geometries
    invalid_mask = ~gdf.is_valid
    if invalid_mask.any():
        invalid_count = int(invalid_mask.sum())
        try:
            gdf.loc[invalid_mask, "geometry"] = gdf.loc[invalid_mask, "geometry"].make_valid()
        except Exception:
            gdf.loc[invalid_mask, "geometry"] = gdf.loc[invalid_mask, "geometry"].buffer(0)
    else:
        invalid_count = 0
    
    # Drop remaining invalid geometries
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty & gdf.is_valid].copy()
    removed = start - len(gdf)
    
    print(f"{name}: start={start}, invalid_before={invalid_count}, removed={removed}, remaining={len(gdf)}")
    return gdf

### Step 3: Load and inspect building data
Load the buildings dataset and display column information.

In [4]:
# Load buildings dataset
buildings = gpd.read_file(buildings_path)

# Clean geometries
buildings = clean_geometries(buildings, "Buildings total")

# Print summary
print(f"Buildings file: {buildings_path}")
print(f"Total buildings after cleaning: {len(buildings)}")
print(f"\nColumn names:")
print(buildings.columns.tolist())

Buildings total: start=629666, invalid_before=178, removed=0, remaining=629666
Buildings file: C:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\data\green_roofs_berlin\buildings_berlin.shp
Total buildings after cleaning: 629666

Column names:
['gml_id', 'importid', 'geb_nutz', 'gruendach', 'ex_int', 'gruen20_m2', 'gint20_m2', 'gex20_m2', 'gruen20_p', 'gint20_p', 'gex20_p', 'geb_area', 'nutz', 'ext', 'egeb_nutz', 'egruendach', 'eex_int', 'geometry']


In [5]:
# Inspect the target columns for underground parking
print("Unique values in 'nutz' column (first 20):")
if "nutz" in buildings.columns:
    print(buildings["nutz"].value_counts(dropna=False).head(20))
else:
    print("Column 'nutz' not found")

print("\n" + "="*50 + "\n")
print("Unique values in 'geb_nutz' column (first 20):")
if "geb_nutz" in buildings.columns:
    print(buildings["geb_nutz"].value_counts(dropna=False).head(20))
else:
    print("Column 'geb_nutz' not found")

Unique values in 'nutz' column (first 20):
nutz
Gebäude (ALKIS)        537417
Gebäude (NOT-ALKIS)     88064
Tiefgarage (ALKIS)       4185
Name: count, dtype: int64


Unique values in 'geb_nutz' column (first 20):
geb_nutz
Wohnen                  322295
Garage, Schuppen        104779
Sonstige                 88064
Bürogebäude, Gewerbe     56090
Nichtwohngebäude         54253
Tiefgarage                4185
Name: count, dtype: int64


### Step 4: Filter out underground parking structures
Remove buildings where 'nutz' or 'geb_nutz' equals 'Tiefgarage' to avoid dataset bias.

In [6]:
# Count buildings marked as Tiefgarage before filtering
tiefgarage_nutz = 0
tiefgarage_geb_nutz = 0
tiefgarage_both = 0

if "nutz" in buildings.columns:
    tiefgarage_nutz = int((buildings["nutz"] == "Tiefgarage").sum())
if "geb_nutz" in buildings.columns:
    tiefgarage_geb_nutz = int((buildings["geb_nutz"] == "Tiefgarage").sum())

print(f"Buildings with nutz='Tiefgarage': {tiefgarage_nutz}")
print(f"Buildings with geb_nutz='Tiefgarage': {tiefgarage_geb_nutz}")

Buildings with nutz='Tiefgarage': 0
Buildings with geb_nutz='Tiefgarage': 4185


In [7]:
# Apply filtering: exclude if either column contains 'Tiefgarage'
buildings_filtered = buildings.copy()

# Filter by nutz column if it exists
if "nutz" in buildings_filtered.columns:
    buildings_filtered = buildings_filtered[buildings_filtered["nutz"] != "Tiefgarage"].copy()

# Filter by geb_nutz column if it exists
if "geb_nutz" in buildings_filtered.columns:
    buildings_filtered = buildings_filtered[buildings_filtered["geb_nutz"] != "Tiefgarage"].copy()

removed_tiefgarage = len(buildings) - len(buildings_filtered)
print(f"Total buildings removed (Tiefgarage): {removed_tiefgarage}")
print(f"Remaining buildings: {len(buildings_filtered)}")

Total buildings removed (Tiefgarage): 4185
Remaining buildings: 625481


### Step 5: Extract green roof indicator and basic features
Identify buildings with green roof potential and compute key attributes.

In [8]:
# Check for green roof related columns
print("Columns in filtered dataset:")
print(buildings_filtered.columns.tolist())

# Look for potential green roof column
green_roof_cols = [c for c in buildings_filtered.columns if "gruendach" in c.lower() or "green" in c.lower() or "dach" in c.lower()]
print(f"\nPotential green roof columns: {green_roof_cols}")

Columns in filtered dataset:
['gml_id', 'importid', 'geb_nutz', 'gruendach', 'ex_int', 'gruen20_m2', 'gint20_m2', 'gex20_m2', 'gruen20_p', 'gint20_p', 'gex20_p', 'geb_area', 'nutz', 'ext', 'egeb_nutz', 'egruendach', 'eex_int', 'geometry']

Potential green roof columns: ['gruendach', 'egruendach']


In [9]:
# Compute roof area
dataset_gdf = buildings_filtered.copy()

# Calculate area in square meters
area_calc = dataset_gdf
if area_calc.crs is not None and area_calc.crs.is_geographic:
    area_calc = area_calc.to_crs(25833)
dataset_gdf["roof_area_m2"] = area_calc.geometry.area.round(2)

# Add a green roof flag (default to unknown, can be refined with actual indicators)
dataset_gdf["has_green_roof"] = False
if "ex_int" in dataset_gdf.columns:
    dataset_gdf["has_green_roof"] = dataset_gdf["ex_int"].notna()
elif "extens" in dataset_gdf.columns:
    dataset_gdf["has_green_roof"] = dataset_gdf["extens"].notna()

print(f"Buildings with potential green roofs: {dataset_gdf['has_green_roof'].sum()}")
print(f"Mean roof area: {dataset_gdf['roof_area_m2'].mean():.2f} m²")

Buildings with potential green roofs: 17229
Mean roof area: 162.91 m²


### Step 6: Prepare and clean export table
Build a clean table with core features for downstream ML models.

In [10]:
# Get all columns except geometry, ensuring has_green_roof and roof_area_m2 are present
export_columns = [c for c in dataset_gdf.columns if c != "geometry"]

print(f"Total columns to export: {len(export_columns)}")
print(f"Sample columns: {export_columns[:10]}")
print(f"\nKey columns check:")
print(f"  - has_green_roof present: {'has_green_roof' in export_columns}")
print(f"  - roof_area_m2 present: {'roof_area_m2' in export_columns}")

Total columns to export: 19
Sample columns: ['gml_id', 'importid', 'geb_nutz', 'gruendach', 'ex_int', 'gruen20_m2', 'gint20_m2', 'gex20_m2', 'gruen20_p', 'gint20_p']

Key columns check:
  - has_green_roof present: True
  - roof_area_m2 present: True


In [11]:
# Build export dataframe (without geometry for CSV)
export_df = dataset_gdf[export_columns].copy()

# Remove rows with missing or zero roof area
export_df = export_df[export_df["roof_area_m2"] > 0].copy()

# Remove duplicate rows
export_df = export_df.drop_duplicates()

print(f"Export table shape: {export_df.shape}")
print(f"Columns retained: {len(export_df.columns)}")
print(f"\nData types:")
print(export_df.dtypes.value_counts())
print(f"\nGreen roof statistics:")
if "has_green_roof" in export_df.columns:
    print(f"  - Buildings with green roof: {export_df['has_green_roof'].sum()}")
if "roof_area_m2" in export_df.columns:
    print(f"  - Mean roof area: {export_df['roof_area_m2'].mean():.2f} m²")
    print(f"  - Max roof area: {export_df['roof_area_m2'].max():.2f} m²")

Export table shape: (625479, 19)
Columns retained: 19

Data types:
str        8
float64    8
int64      2
bool       1
Name: count, dtype: int64

Green roof statistics:
  - Buildings with green roof: 17229
  - Mean roof area: 162.91 m²
  - Max roof area: 69736.16 m²


### Step 7: Export cleaned data
Save as CSV, GPKG, and SHP for use in downstream pipelines.

In [ ]:
# Set up export directory
output_dir = project_root / "data" / "exports" / "preprocessing_step0_gr_roof_cl"
output_dir.mkdir(parents=True, exist_ok=True)

output_csv = output_dir / "buildings_cleaned_no_tiefgarage.csv"
output_gpkg = output_dir / "buildings_cleaned_no_tiefgarage.gpkg"
output_shp = output_dir / "buildings_cleaned_no_tiefgarage.shp"

# Export CSV (no geometry)
export_df.to_csv(output_csv, index=False)
print(f"CSV saved: {output_csv}")

CSV saved: C:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\data\exports\processing_step0_gr_roof_cl\buildings_cleaned_no_tiefgarage.csv


In [13]:
# Export GeoPackage (with geometry and all features)
export_gdf = dataset_gdf[export_columns + ["geometry"]].copy()
export_gdf = export_gdf[export_gdf["roof_area_m2"] > 0].copy()

export_gdf.to_file(output_gpkg, driver="GPKG", layer="buildings_cleaned")
print(f"GPKG saved: {output_gpkg} ({len(export_gdf)} features)")
print(f"  - Columns in export: {len(export_gdf.columns)}")

GPKG saved: C:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\data\exports\processing_step0_gr_roof_cl\buildings_cleaned_no_tiefgarage.gpkg (625479 features)
  - Columns in export: 20


In [14]:
# Export Shapefile (polygon geometries only)
polygon_types = {"Polygon", "MultiPolygon"}
export_gdf_poly = export_gdf[export_gdf.geometry.geom_type.isin(polygon_types)].copy()

if len(export_gdf_poly) > 0:
    export_gdf_poly.to_file(output_shp)
    print(f"SHP saved: {output_shp} ({len(export_gdf_poly)} features)")
else:
    print("No SHP exported: no polygon geometries found.")

C:\Users\elbma\AppData\Local\Temp\ipykernel_30044\1769791219.py:6: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  export_gdf_poly.to_file(output_shp)


SHP saved: C:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\data\exports\processing_step0_gr_roof_cl\buildings_cleaned_no_tiefgarage.shp (625479 features)


c:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'roof_area_m2' to 'roof_area_'
  ogr_write(
c:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'has_green_roof' to 'has_green_'
  ogr_write(


### Step 8: Summary and Quality Metrics
Print final quality report.

In [15]:
print("\n" + "="*70)
print("SUMMARY: Green Roof Data Cleaning & Preparation")
print("="*70)
print(f"\nInput Data:")
print(f"  - Original building count: {len(buildings)}")
print(f"  - Buildings removed (Tiefgarage): {removed_tiefgarage}")
print(f"  - Remaining buildings after filter: {len(buildings_filtered)}")
print(f"\nOutput Data:")
print(f"  - Buildings with roof area > 0: {len(export_gdf)}")
print(f"  - Buildings with green roof indicator: {export_gdf['has_green_roof'].sum()}")
print(f"  - Total columns in export: {len(export_columns)}")
print(f"\nColumn Coverage (preserved from original dataset):")
print(f"  - Green roof features: gruen20_m2, gint20_m2, gint20_p, gex20_m2, gex20_p")
print(f"  - Building properties: nutz, geb_nutz, bauweise, ist_denkma, anzahl_unt, anzahl_obe")
print(f"  - New feature: has_green_roof (bool), roof_area_m2 (float)")
print(f"\nExport Files:")
print(f"  - Directory: {output_dir}")
print(f"  - CSV: {output_csv.name} ({len(export_df)} rows, {len(export_df.columns)} columns)")
print(f"  - GPKG: {output_gpkg.name} ({len(export_gdf)} features, with geometry)")
print(f"  - SHP: {output_shp.name} (polygon geometries only)")
print("="*70)


SUMMARY: Green Roof Data Cleaning & Preparation

Input Data:
  - Original building count: 629666
  - Buildings removed (Tiefgarage): 4185
  - Remaining buildings after filter: 625481

Output Data:
  - Buildings with roof area > 0: 625479
  - Buildings with green roof indicator: 17229
  - Total columns in export: 19

Column Coverage (preserved from original dataset):
  - Green roof features: gruen20_m2, gint20_m2, gint20_p, gex20_m2, gex20_p
  - Building properties: nutz, geb_nutz, bauweise, ist_denkma, anzahl_unt, anzahl_obe
  - New feature: has_green_roof (bool), roof_area_m2 (float)

Export Files:
  - Directory: C:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\data\exports\processing_step0_gr_roof_cl
  - CSV: buildings_cleaned_no_tiefgarage.csv (625479 rows, 19 columns)
  - GPKG: buildings_cleaned_no_tiefgarage.gpkg (625479 features, with geometry)
  - SHP: buildings_cleaned_no_tiefgarage.shp (polygon geometries only)
